In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -y -qq install tshark
!tshark -v | head -n 3

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package libpcap0.8:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-libpcap0.8_1.10.1-4ubuntu1.22.04.1_amd64.deb ...
Unpacking libpcap0.8:amd64 (1.10.1-4ubuntu1.22.04.1) ...
Selecting previously unselected package libbcg729-0:amd64.
Preparing to unpack .../01-libbcg729-0_1.1.1-2_amd64.deb ...
Unpacking libbcg729-0:amd64 (1.1.1-2) ...
Selecting previously unselected package liblua5.2-0:amd64.
Preparing to unpack .../02-liblua5.2-0_5.2.4-2_amd64.deb ...
Unpacking liblua5.2-0:amd64 (5.2.4-2) ...
Selecting previously unselected package libnl-genl-3-200:amd64.
Preparing to unpack .../03-libnl-genl-3-200_3.5.0-0.1_amd64.deb ...
Unpacking libnl-genl-3-200:amd64 (3.5.0-0.1) ...
Selecting prev

In [2]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/iot/iot device name/network")
PINH = BASE / "Pinh19"
OKUI = BASE / "Okui22"

folders = ["raw_packet_csvs", "labeled_packet_csvs", "feature_csvs", "results"]

for name in folders:
    folder = PINH / name
    folder.mkdir(parents=True, exist_ok=True)

print("ready:", PINH)

ready: /content/drive/MyDrive/iot/iot device name/network/Pinh19


In [3]:
src_map = OKUI / "device_map.csv"
dst_map = PINH / "device_map.csv"

if src_map.exists():
    shutil.copy2(src_map, dst_map)
    print("copied:", dst_map)
else:
    print("device_map.csv not found")

copied: /content/drive/MyDrive/iot/iot device name/network/Pinh19/device_map.csv


In [4]:
import pandas as pd
import numpy as np
import subprocess
from IPython.display import display

In [5]:
map_path = PINH / "device_map.csv"
device_map = pd.read_csv(map_path)

device_map["ip"] = device_map["ip"].fillna("").astype(str).str.strip()
device_map["best_mac"] = device_map["best_mac"].fillna("").astype(str).str.strip().str.lower()
device_map["service_name"] = device_map["service_name"].fillna("").astype(str).str.strip()

display(device_map.head(20))
print(device_map.columns.tolist())

,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


['ip', 'best_mac', 'service_name']


In [6]:
mac_to_device = {}
ip_to_device = {}

for i in range(len(device_map)):
    row = device_map.iloc[i]

    mac = row["best_mac"]
    ip = row["ip"]
    name = row["service_name"]

    if mac != "" and name != "":
        if mac not in mac_to_device:
            mac_to_device[mac] = name

    if ip != "" and name != "":
        if ip not in ip_to_device:
            ip_to_device[ip] = name

print("MAC labels:", len(mac_to_device))
print("IP labels :", len(ip_to_device))

MAC labels: 7
IP labels : 12


In [7]:
def save_packets(pcap_path, out_csv):
    fields = [
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
        "ip.src",
        "ip.dst",
        "ip.proto",
        "tcp.srcport",
        "tcp.dstport",
        "udp.srcport",
        "udp.dstport",
    ]

    cmd = ["tshark", "-r", str(pcap_path), "-T", "fields"]

    for f in fields:
        cmd += ["-e", f]

    cmd += ["-E", "header=y"]
    cmd += ["-E", "separator=,"]
    cmd += ["-E", "quote=d"]
    cmd += ["-E", "occurrence=f"]

    with open(out_csv, "w", encoding="utf-8") as f:
        subprocess.run(cmd, stdout=f, check=True)

    print("saved:", out_csv)

In [ ]:
def label_packets(raw_csv_path, labeled_csv_path, mac_to_device, ip_to_device):
    df = pd.read_csv(raw_csv_path, low_memory=False)

    need_cols = ["eth.src", "eth.dst", "ip.src", "ip.dst"]
    for col in need_cols:
        if col not in df.columns:
            df[col] = ""

    for col in ["eth.src", "eth.dst"]:
        df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

    for col in ["ip.src", "ip.dst"]:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df["src_device"] = df["eth.src"].map(mac_to_device)
    miss_src = df["src_device"].isna()
    df.loc[miss_src, "src_device"] = df.loc[miss_src, "ip.src"].map(ip_to_device)

    df["dst_device"] = df["eth.dst"].map(mac_to_device)
    miss_dst = df["dst_device"].isna()
    df.loc[miss_dst, "dst_device"] = df.loc[miss_dst, "ip.dst"].map(ip_to_device)

    
    df["tx_device"] = df["src_device"]

    df.to_csv(labeled_csv_path, index=False)
    print("saved:", labeled_csv_path)

    show_cols = ["eth.src", "ip.src", "src_device", "eth.dst", "ip.dst", "dst_device", "tx_device"]
    show_cols = [c for c in show_cols if c in df.columns]
    display(df[show_cols].head(10))

    return df

In [9]:
def build_pinh_features(labeled_csv_path, feature_csv_path):
    df = pd.read_csv(labeled_csv_path, low_memory=False)

    if "tx_device" not in df.columns:
        print("tx_device column not found")
        return pd.DataFrame()

    df = df[df["tx_device"].notna()].copy()

    df["frame.time_epoch"] = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df["frame.len"] = pd.to_numeric(df["frame.len"], errors="coerce")

    df = df.dropna(subset=["frame.time_epoch", "frame.len"]).copy()

    if len(df) == 0:
        out = pd.DataFrame(columns=[
            "device_name", "window_start",
            "total_tx_bytes", "mean_tx_bytes", "std_tx_bytes",
            "source_file"
        ])
        out.to_csv(feature_csv_path, index=False)
        return out

    df["ts"] = pd.to_datetime(df["frame.time_epoch"], unit="s", utc=True)
    df["window_start"] = df["ts"].dt.floor("1s")

    rows = []

    grouped = df.groupby(["tx_device", "window_start"], sort=True)

    for (device_name, window_start), part in grouped:
        values = part["frame.len"].tolist()

        total_tx_bytes = 0.0
        for v in values:
            total_tx_bytes += float(v)

        mean_tx_bytes = 0.0
        if len(values) > 0:
            mean_tx_bytes = total_tx_bytes / len(values)

        std_tx_bytes = 0.0
        if len(values) > 1:
            s = 0.0
            for v in values:
                diff = float(v) - mean_tx_bytes
                s += diff * diff
            std_tx_bytes = (s / len(values)) ** 0.5

        rows.append({
            "device_name": device_name,
            "window_start": window_start,
            "total_tx_bytes": total_tx_bytes,
            "mean_tx_bytes": mean_tx_bytes,
            "std_tx_bytes": std_tx_bytes,
        })

    feat_df = pd.DataFrame(rows)

    source_name = Path(labeled_csv_path).stem
    source_name = source_name.replace("_packets_labeled", "")
    feat_df["source_file"] = source_name

    feat_df.to_csv(feature_csv_path, index=False)
    print("saved:", feature_csv_path)

    display(feat_df.head())
    print(feat_df.shape)
    print(feat_df["device_name"].value_counts())

    return feat_df

Let's start by capturing a PCAP file to test it.

In [12]:
pcap_path = BASE / "normal_1.pcap"
raw_csv_path = PINH / "raw_packet_csvs" / "normal_1_packets.csv"
labeled_csv_path = PINH / "labeled_packet_csvs" / "normal_1_packets_labeled.csv"
feature_csv_path = PINH / "feature_csvs" / "normal_1_pinh_features.csv"

save_packets(pcap_path, raw_csv_path)

raw_df = pd.read_csv(raw_csv_path, low_memory=False)
display(raw_df.head())
print(raw_df.shape)

labeled_df = label_packets(raw_csv_path, labeled_csv_path, mac_to_device, ip_to_device)
print(labeled_df["tx_device"].value_counts(dropna=False).head(20))

feat_df = build_pinh_features(labeled_csv_path, feature_csv_path)

saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/raw_packet_csvs/normal_1_packets.csv


,frame.time_epoch,frame.len,eth.src,eth.dst,ip.src,ip.dst,ip.proto,tcp.srcport,tcp.dstport,udp.srcport,udp.dstport
0,1.554220e+09,551,00:00:7f:06:ed:8c,09:00:02:17:04:d2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.554220e+09,552,00:00:7f:06:ec:89,0a:00:02:18:04:d2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.554220e+09,189,NaN,NaN,192.168.1.152,192.168.1.192,6.0,1880.0,40571.0,NaN,NaN
3,1.554220e+09,189,NaN,NaN,192.168.1.152,192.168.1.190,6.0,1880.0,43539.0,NaN,NaN
4,1.554220e+09,68,NaN,NaN,192.168.1.190,192.168.1.152,6.0,43539.0,1880.0,NaN,NaN


(2000000, 11)
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/labeled_packet_csvs/normal_1_packets_labeled.csv


,eth.src,ip.src,src_device,eth.dst,ip.dst,dst_device,tx_device
0,00:00:7f:06:ed:8c,,NaN,09:00:02:17:04:d2,,NaN,NaN
1,00:00:7f:06:ec:89,,NaN,0a:00:02:18:04:d2,,NaN,NaN
2,,192.168.1.152,service_1,,192.168.1.192,service_2,service_1
3,,192.168.1.152,service_1,,192.168.1.190,service_3,service_1
4,,192.168.1.190,service_3,,192.168.1.152,service_1,service_3
5,,192.168.1.152,service_1,,192.168.1.195,NaN,service_1
6,,192.168.1.195,NaN,,192.168.1.152,service_1,NaN
7,00:00:7f:06:eb:86,,NaN,0b:00:02:19:04:d2,,NaN,NaN
8,,192.168.1.192,service_2,,192.168.1.152,service_1,service_2
9,00:00:7f:06:ea:83,,NaN,0c:00:02:1a:04:d2,,NaN,NaN


tx_device
NaN          1652500
service_1     183656
service_3      99845
service_2      63484
service_6        266
service_7        158
service_4         91
Name: count, dtype: int64
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/feature_csvs/normal_1_pinh_features.csv


,device_name,window_start,total_tx_bytes,mean_tx_bytes,std_tx_bytes,source_file
0,service_1,2019-04-02 15:52:04+00:00,5073.0,563.666667,340.233580,normal_1
1,service_1,2019-04-02 15:52:05+00:00,17156.0,612.714286,251.291359,normal_1
2,service_1,2019-04-02 15:52:06+00:00,21436.0,612.457143,411.658898,normal_1
3,service_1,2019-04-02 15:52:07+00:00,26608.0,760.228571,691.398834,normal_1
4,service_1,2019-04-02 15:52:08+00:00,6496.0,499.692308,284.248370,normal_1


(29721, 6)
device_name
service_3    9803
service_2    9800
service_1    9797
service_6     191
service_7     107
service_4      23
Name: count, dtype: int64


In [13]:
pcap_files = []

for i in range(1, 14):
    p = BASE / f"normal_{i}.pcap"
    if p.exists():
        pcap_files.append(p)

print("num pcaps found:", len(pcap_files))
for p in pcap_files:
    print("-", p.name)

num pcaps found: 13
- normal_1.pcap
- normal_2.pcap
- normal_3.pcap
- normal_4.pcap
- normal_5.pcap
- normal_6.pcap
- normal_7.pcap
- normal_8.pcap
- normal_9.pcap
- normal_10.pcap
- normal_11.pcap
- normal_12.pcap
- normal_13.pcap


In [14]:
summary_rows = []

for pcap_path in pcap_files:
    stem = pcap_path.stem

    raw_csv_path = PINH / "raw_packet_csvs" / f"{stem}_packets.csv"
    labeled_csv_path = PINH / "labeled_packet_csvs" / f"{stem}_packets_labeled.csv"
    feature_csv_path = PINH / "feature_csvs" / f"{stem}_pinh_features.csv"

    print("\n=== processing", stem, "===")

    # 1) pcap -> raw csv
    if not raw_csv_path.exists():
        save_packets(pcap_path, raw_csv_path)
    else:
        print("raw exists:", raw_csv_path.name)

    # 2) raw -> labeled
    labeled_df = label_packets(raw_csv_path, labeled_csv_path, mac_to_device, ip_to_device)

    tx_nonnull = labeled_df["tx_device"].notna().sum()
    print("tx_device non-null:", tx_nonnull)

    # 3) labeled -> feature
    feat_df = build_pinh_features(labeled_csv_path, feature_csv_path)

    class_count = {}
    if len(feat_df) > 0:
        vc = feat_df["device_name"].value_counts()
        for name in vc.index:
            class_count[f"class_{name}"] = int(vc[name])

    row = {
        "pcap": stem,
        "raw_rows": len(labeled_df),
        "tx_device_nonnull_rows": int(tx_nonnull),
        "feature_rows": len(feat_df),
        "n_classes": int(feat_df["device_name"].nunique()) if len(feat_df) > 0 else 0
    }

    for k in class_count:
        row[k] = class_count[k]

    summary_rows.append(row)

print("\nall done")


=== processing normal_1 ===
raw exists: normal_1_packets.csv
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/labeled_packet_csvs/normal_1_packets_labeled.csv


,eth.src,ip.src,src_device,eth.dst,ip.dst,dst_device,tx_device
0,00:00:7f:06:ed:8c,,NaN,09:00:02:17:04:d2,,NaN,NaN
1,00:00:7f:06:ec:89,,NaN,0a:00:02:18:04:d2,,NaN,NaN
2,,192.168.1.152,service_1,,192.168.1.192,service_2,service_1
3,,192.168.1.152,service_1,,192.168.1.190,service_3,service_1
4,,192.168.1.190,service_3,,192.168.1.152,service_1,service_3
5,,192.168.1.152,service_1,,192.168.1.195,NaN,service_1
6,,192.168.1.195,NaN,,192.168.1.152,service_1,NaN
7,00:00:7f:06:eb:86,,NaN,0b:00:02:19:04:d2,,NaN,NaN
8,,192.168.1.192,service_2,,192.168.1.152,service_1,service_2
9,00:00:7f:06:ea:83,,NaN,0c:00:02:1a:04:d2,,NaN,NaN


tx_device non-null: 347500
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/feature_csvs/normal_1_pinh_features.csv


,device_name,window_start,total_tx_bytes,mean_tx_bytes,std_tx_bytes,source_file
0,service_1,2019-04-02 15:52:04+00:00,5073.0,563.666667,340.233580,normal_1
1,service_1,2019-04-02 15:52:05+00:00,17156.0,612.714286,251.291359,normal_1
2,service_1,2019-04-02 15:52:06+00:00,21436.0,612.457143,411.658898,normal_1
3,service_1,2019-04-02 15:52:07+00:00,26608.0,760.228571,691.398834,normal_1
4,service_1,2019-04-02 15:52:08+00:00,6496.0,499.692308,284.248370,normal_1


(29721, 6)
device_name
service_3    9803
service_2    9800
service_1    9797
service_6     191
service_7     107
service_4      23
Name: count, dtype: int64

=== processing normal_2 ===
raw exists: normal_2_packets.csv
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/labeled_packet_csvs/normal_2_packets_labeled.csv


...

(24574, 6)
device_name
service_1    11452
service_3    11452
service_4     1579
service_7       76
service_5       11
service_6        4
Name: count, dtype: int64

=== processing normal_13 ===
raw exists: normal_13_packets.csv
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/labeled_packet_csvs/normal_13_packets_labeled.csv


,eth.src,ip.src,src_device,eth.dst,ip.dst,dst_device,tx_device
0,00:00:7f:06:e8:4a,,NaN,0f:00:01:4d:04:d2,,NaN,NaN
1,00:00:7f:06:f6:67,,NaN,00:00:01:4e:04:d2,,NaN,NaN
2,00:00:7f:06:f5:64,,NaN,01:00:01:4f:04:d2,,NaN,NaN
3,00:00:7f:06:f4:61,,NaN,02:00:01:50:04:d2,,NaN,NaN
4,00:00:7f:06:f3:5e,,NaN,03:00:01:51:04:d2,,NaN,NaN
5,00:00:7f:06:f2:5b,,NaN,04:00:01:52:04:d2,,NaN,NaN
6,00:00:7f:06:f1:58,,NaN,05:00:01:53:04:d2,,NaN,NaN
7,00:00:7f:06:f0:55,,NaN,06:00:01:54:04:d2,,NaN,NaN
8,,,NaN,,,NaN,NaN
9,00:00:7f:06:ef:52,,NaN,07:00:01:55:04:d2,,NaN,NaN


tx_device non-null: 171053
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/feature_csvs/normal_13_pinh_features.csv


,device_name,window_start,total_tx_bytes,mean_tx_bytes,std_tx_bytes,source_file
0,service_1,2019-04-03 06:47:20+00:00,189.0,189.000000,0.000000,normal_13
1,service_1,2019-04-03 06:47:21+00:00,2542.0,363.142857,329.031231,normal_13
2,service_1,2019-04-03 06:47:22+00:00,8661.0,787.363636,938.579030,normal_13
3,service_1,2019-04-03 06:47:23+00:00,7294.0,810.444444,668.370508,normal_13
4,service_1,2019-04-03 06:47:24+00:00,8508.0,945.333333,846.225213,normal_13


(31352, 6)
device_name
service_1    10674
service_3    10673
service_4     4935
service_2     4440
service_5      320
service_7      196
service_6      114
Name: count, dtype: int64

all done


In [15]:
summary_df = pd.DataFrame(summary_rows)
summary_path = PINH / "results" / "feature_build_summary.csv"

summary_df.to_csv(summary_path, index=False)

print("saved:", summary_path)
display(summary_df)

saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/feature_build_summary.csv


,pcap,raw_rows,tx_device_nonnull_rows,feature_rows,n_classes,class_service_3,class_service_2,class_service_1,class_service_6,class_service_7,class_service_4,class_service_5
0,normal_1,2000000,347500,29721,6,9803,9800.0,9797,191.0,107.0,23.0,NaN
1,normal_2,2000000,206439,33090,6,10877,10875.0,10877,53.0,316.0,92.0,NaN
2,normal_3,2000000,197931,33463,5,11146,11057.0,11148,42.0,70.0,NaN,NaN
3,normal_4,2000000,215893,36799,7,10967,10663.0,10967,206.0,171.0,3482.0,343.0
4,normal_5,2000000,207349,39063,7,10925,10447.0,10925,93.0,226.0,6066.0,381.0
5,normal_6,2000000,204743,39253,7,10897,9932.0,10803,187.0,251.0,6714.0,469.0
6,normal_7,2000000,145919,26970,5,11302,NaN,11302,587.0,228.0,3551.0,NaN
7,normal_8,2000000,136778,22754,5,11229,NaN,11230,196.0,85.0,14.0,NaN
8,normal_9,2000000,135699,23085,5,11472,NaN,11473,53.0,63.0,24.0,NaN
9,normal_10,2000000,138148,23109,4,11516,NaN,11516,45.0,NaN,32.0,NaN


In [16]:
feature_files = sorted((PINH / "feature_csvs").glob("*_pinh_features.csv"))
print("feature files:", len(feature_files))

all_parts = []
for f in feature_files:
    part = pd.read_csv(f)
    all_parts.append(part)

all_feat = pd.concat(all_parts, ignore_index=True)

all_feat["window_start"] = pd.to_datetime(all_feat["window_start"], errors="coerce", utc=True)
all_feat = all_feat.dropna(subset=["window_start", "device_name"]).copy()

all_feat_path = PINH / "results" / "all_pinh_features.csv"
all_feat.to_csv(all_feat_path, index=False)

print("saved:", all_feat_path)
display(all_feat.head())
print(all_feat.shape)
print(all_feat["device_name"].value_counts())

feature files: 13
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/all_pinh_features.csv


,device_name,window_start,total_tx_bytes,mean_tx_bytes,std_tx_bytes,source_file
0,service_1,2019-04-03 20:38:05+00:00,13778.0,1252.545455,1473.409612,normal_10
1,service_1,2019-04-03 20:38:06+00:00,5534.0,691.750000,913.910383,normal_10
2,service_1,2019-04-03 20:38:07+00:00,2655.0,379.285714,285.521435,normal_10
3,service_1,2019-04-03 20:38:08+00:00,1585.0,264.166667,299.545443,normal_10
4,service_1,2019-04-03 20:38:09+00:00,5826.0,728.250000,862.503442,normal_10


(386223, 6)
device_name
service_3    143726
service_1    143632
service_2     67214
service_4     26567
service_7      1789
service_6      1771
service_5      1524
Name: count, dtype: int64


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score, classification_report

In [18]:
df = pd.read_csv(PINH / "results" / "all_pinh_features.csv")

df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce", utc=True)

for col in ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[
    "window_start",
    "device_name",
    "source_file",
    "total_tx_bytes",
    "mean_tx_bytes",
    "std_tx_bytes"
]).copy()

FEATURES = ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]

X = df[FEATURES]
y = df["device_name"]
groups = df["source_file"]

print(df.shape)
print(df["device_name"].value_counts())

(386223, 6)
device_name
service_3    143726
service_1    143632
service_2     67214
service_4     26567
service_7      1789
service_6      1771
service_5      1524
Name: count, dtype: int64


In [19]:
gkf = GroupKFold(n_splits=7)
target_fold = 4

for fold_id, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    if fold_id != target_fold:
        continue

    X_train = X.iloc[tr_idx]
    X_test = X.iloc[te_idx]

    y_train = y.iloc[tr_idx]
    y_test = y.iloc[te_idx]

    train_files = sorted(df.iloc[tr_idx]["source_file"].unique())
    test_files = sorted(df.iloc[te_idx]["source_file"].unique())

    print("Fold:", fold_id)
    print("train files:", train_files)
    print("test files :", test_files)
    print()
    print("train shape:", X_train.shape)
    print("test shape :", X_test.shape)
    print()
    print("train classes:")
    print(y_train.value_counts())
    print()
    print("test classes:")
    print(y_test.value_counts())

    clf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)
    classes = clf.classes_

    y_bin = label_binarize(y_test, classes=classes)

    if len(classes) == 2 and y_bin.shape[1] == 1:
        y_bin = np.hstack([1 - y_bin, y_bin])

    macro_aucpr = average_precision_score(y_bin, proba, average="macro")
    weighted_aucpr = average_precision_score(y_bin, proba, average="weighted")
    micro_aucpr = average_precision_score(y_bin, proba, average="micro")

    print()
    print("Macro AUCPR   :", macro_aucpr)
    print("Weighted AUCPR:", weighted_aucpr)
    print("Micro AUCPR   :", micro_aucpr)
    print()
    print(classification_report(y_test, pred, zero_division=0))

    break

Fold: 4
train files: ['normal_1', 'normal_10', 'normal_11', 'normal_12', 'normal_13', 'normal_2', 'normal_4', 'normal_5', 'normal_6', 'normal_7', 'normal_8']
test files : ['normal_3', 'normal_9']

train shape: (329675, 3)
test shape : (56548, 3)

train classes:
device_name
service_3    121108
service_1    121011
service_2     56157
service_4     26543
service_6      1676
service_7      1656
service_5      1524
Name: count, dtype: int64

test classes:
device_name
service_1    22621
service_3    22618
service_2    11057
service_7      133
service_6       95
service_4       24
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(



Macro AUCPR   : 0.8400004806248498
Weighted AUCPR: 0.99990128514576
Micro AUCPR   : 0.9999829021789997

              precision    recall  f1-score   support

   service_1       1.00      1.00      1.00     22621
   service_2       1.00      1.00      1.00     11057
   service_3       1.00      1.00      1.00     22618
   service_4       0.77      0.83      0.80        24
   service_5       0.00      0.00      0.00         0
   service_6       0.88      0.96      0.92        95
   service_7       0.98      0.90      0.94       133

    accuracy                           1.00     56548
   macro avg       0.80      0.81      0.81     56548
weighted avg       1.00      1.00      1.00     56548



In [20]:



path = "/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/all_pinh_features.csv"
df = pd.read_csv(path)


df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce", utc=True)
for c in ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[
    "window_start", "device_name", "source_file",
    "total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"
]).copy()



hours = df["window_start"].dt.hour
df["mode"] = np.where((hours >= 2) & (hours < 6), "active", "idle")

print(df["mode"].value_counts())
print(df[["window_start", "device_name", "source_file", "mode"]].head())

FEATURES = ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]
X = df[FEATURES]
y = df["device_name"]
groups = df["source_file"]

def multiclass_aucpr(y_true, proba, classes):
    Y = label_binarize(y_true, classes=classes)
    if len(classes) == 2 and Y.shape[1] == 1:
        Y = np.hstack([1 - Y, Y])

    macro = average_precision_score(Y, proba, average="macro")
    weighted = average_precision_score(Y, proba, average="weighted")
    micro = average_precision_score(Y, proba, average="micro")
    return macro, weighted, micro

def eval_on_subset(clf, test_df, scenario_name):
    """
    scenario_name: idle / active / mix
    """
    if scenario_name == "mix":
        sub = test_df.copy()
    else:
        sub = test_df[test_df["mode"] == scenario_name].copy()

    if len(sub) == 0:
        return None

    X_test = sub[FEATURES]
    y_test = sub["device_name"]

  
    seen_classes = clf.classes_
    mask = y_test.isin(seen_classes)
    sub = sub[mask].copy()

    if len(sub) == 0:
        return None

    X_test = sub[FEATURES]
    y_test = sub["device_name"]

    
    if y_test.nunique() < 2:
        return None

    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)
    classes = clf.classes_

    macro_aucpr, weighted_aucpr, micro_aucpr = multiclass_aucpr(y_test, proba, classes)

    return {
        "scenario": scenario_name,
        "n_test": len(sub),
        "n_classes_test": y_test.nunique(),
        "macro_aucpr": macro_aucpr,
        "weighted_aucpr": weighted_aucpr,
        "micro_aucpr": micro_aucpr,
        "report": classification_report(y_test, pred, zero_division=0)
    }

# ========= GroupKFold: Train=Mix, Test=Idle/Active/Mix =========
gkf = GroupKFold(n_splits=10)

rows = []
reports = {}

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    train_df = df.iloc[tr_idx].copy()
    test_df = df.iloc[te_idx].copy()

    X_train = train_df[FEATURES]
    y_train = train_df["device_name"]

    clf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    train_files = sorted(train_df["source_file"].unique())
    test_files = sorted(test_df["source_file"].unique())

    for scenario in ["idle", "active", "mix"]:
        res = eval_on_subset(clf, test_df, scenario)
        if res is None:
            continue

        rows.append({
            "fold": fold,
            "scenario": scenario,
            "n_train": len(train_df),
            "n_test": res["n_test"],
            "n_classes_train": y_train.nunique(),
            "n_classes_test": res["n_classes_test"],
            "train_files": len(train_files),
            "test_files": len(test_files),
            "train_file_names": ", ".join(train_files),
            "test_file_names": ", ".join(test_files),
            "macro_aucpr": res["macro_aucpr"],
            "weighted_aucpr": res["weighted_aucpr"],
            "micro_aucpr": res["micro_aucpr"],
        })

        reports[(fold, scenario)] = res["report"]

mode_cv_df = pd.DataFrame(rows)
display(mode_cv_df)

# ========= average =========
summary = (
    mode_cv_df.groupby("scenario", as_index=False)
              .agg(
                  folds=("fold", "count"),
                  macro_aucpr_mean=("macro_aucpr", "mean"),
                  macro_aucpr_std=("macro_aucpr", lambda s: s.std(ddof=0)),
                  weighted_aucpr_mean=("weighted_aucpr", "mean"),
                  weighted_aucpr_std=("weighted_aucpr", lambda s: s.std(ddof=0)),
                  micro_aucpr_mean=("micro_aucpr", "mean"),
                  micro_aucpr_std=("micro_aucpr", lambda s: s.std(ddof=0)),
              )
)

display(summary)


out1 = "/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_fold_results.csv"
out2 = "/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_summary.csv"

mode_cv_df.to_csv(out1, index=False)
summary.to_csv(out2, index=False)

print("saved:", out1)
print("saved:", out2)

mode
idle      304186
active     82037
Name: count, dtype: int64
               window_start device_name source_file  mode
0 2019-04-03 20:38:05+00:00   service_1   normal_10  idle
1 2019-04-03 20:38:06+00:00   service_1   normal_10  idle
2 2019-04-03 20:38:07+00:00   service_1   normal_10  idle
3 2019-04-03 20:38:08+00:00   service_1   normal_10  idle
4 2019-04-03 20:38:09+00:00   service_1   normal_10  idle


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

,fold,scenario,n_train,n_test,n_classes_train,n_classes_test,train_files,test_files,train_file_names,test_file_names,macro_aucpr,weighted_aucpr,micro_aucpr
0,1,idle,346970,39253,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_6,0.973407,0.998424,0.999235
1,1,mix,346970,39253,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_6,0.973407,0.998424,0.999235
2,2,idle,347160,9795,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_5,0.968132,0.998124,0.998945
3,2,active,347160,29268,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_5,0.995326,0.999586,0.999869
4,2,mix,347160,39063,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_5,0.988718,0.999182,0.999638
5,3,idle,349424,14353,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_4,0.953317,0.997147,0.998664
6,3,active,349424,22446,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_4,0.968368,0.998813,0.999910
7,3,mix,349424,36799,7,7,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_4,0.948767,0.997745,0.999427
8,4,idle,352760,33463,7,5,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_3,0.706686,0.999925,0.999998
9,4,mix,352760,33463,7,5,12,1,"normal_1, normal_10, normal_11, normal_12, nor...",normal_3,0.706686,0.999925,0.999998


,scenario,folds,macro_aucpr_mean,macro_aucpr_std,weighted_aucpr_mean,weighted_aucpr_std,micro_aucpr_mean,micro_aucpr_std
0,active,3,0.844495,0.194557,0.999404,0.000428,0.999863,0.000041
1,idle,10,0.822126,0.130343,0.998697,0.001072,0.999203,0.000765
2,mix,10,0.834509,0.122464,0.998862,0.000981,0.999337,0.000733


saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_fold_results.csv
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_summary.csv
